In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# ----- Edit these paths for your environment -----
# DATA_PATH: the AnnData file (log2(TMM-CPM+1) gene expression matrix used for SVM features).
# RESULT_DIR: where the train_*_svm.py scripts wrote their CSV outputs.
# PLOT_DIR:   where to save the figures generated by this notebook.
DATA_PATH = "data/75k_unstable_anndata_log2tmmcpm_classification.h5ad"
RESULT_DIR = "results"
PLOT_DIR = "plots"

# Filename prefixes produced by the train_*_svm.py scripts (their default --metrics-prefix)
MULTICLASS_RBF_PREFIX = "multiclass_rbf_svm"
BINARY_RBF_PREFIX     = "binary_rbf_svm"

In [ ]:
import os

os.makedirs(PLOT_DIR, exist_ok=True)

cm_path     = f"{RESULT_DIR}/{MULTICLASS_RBF_PREFIX}_confusion_matrix.csv"
scores_path = f"{RESULT_DIR}/{MULTICLASS_RBF_PREFIX}_scores.csv"

cm_df = pd.read_csv(cm_path, index_col=0)
scores_df = pd.read_csv(scores_path)

# Class counts from ground truth in scores
counts = scores_df["y_true"].value_counts()

class_cols = [c for c in scores_df.columns if c not in ("y_true", "y_pred")]
# Top 5 non-no_infection classes by cell count, then add no_infection if present
non_no = [c for c in counts.index if c != "no_infection" and c in class_cols]
top5 = non_no[:5]
classes_to_plot = list(top5)
if "no_infection" in counts.index:
    classes_to_plot.append("no_infection")

In [3]:
scores_df

,Unnamed: 0,y_true,y_pred,Cyprinid herpesvirus 3,Danio blood picornavirus_Zebrafish picornavirus 2,Influenza Z virus,Largemouth bass virus,Redspotted grouper nervous necrosis virus,Rocky Mountain birnavirus,Spring viraemia of carp virus,Zebrafish gut calicivirus,Zebrafish jaw picornavirus,Zebrafish jaw poxvirus,Zebrafish picornavirus 1,Zebrafish rhabdo-like virus,Zebrafish systemic calicivirus,no_infection
0,ERR830291,no_infection,no_infection,0.000063,0.000795,0.000030,0.000012,1.824824e-06,0.000006,0.000002,0.000008,0.000007,0.001357,0.000568,0.000047,0.000003,0.997098
1,SRR24036760,no_infection,no_infection,0.000041,0.002144,0.000014,0.000334,3.716919e-05,0.000004,0.000020,0.000030,0.000397,0.001068,0.004589,0.000016,0.000003,0.991304
2,SRR22258547,no_infection,Zebrafish jaw poxvirus,0.000389,0.152892,0.000151,0.000287,3.432481e-05,0.000056,0.000080,0.000174,0.000901,0.352421,0.008513,0.000581,0.000053,0.483469
3,ERR1216124,no_infection,no_infection,0.000241,0.000405,0.000030,0.000015,8.531790e-07,0.000006,0.000001,0.000013,0.000009,0.001027,0.000982,0.000020,0.000003,0.997249
4,ERR738220,no_infection,no_infection,0.000107,0.000181,0.000020,0.000015,9.805931e-07,0.000003,0.000001,0.000007,0.000024,0.001861,0.000690,0.000006,0.000004,0.997082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12252,ERR1215810,no_infection,no_infection,0.000253,0.001437,0.000047,0.000023,1.444004e-06,0.000011,0.000004,0.000026,0.000018,0.001504,0.001342,0.000077,0.000003,0.995253
12253,SRR31539738,no_infection,no_infection,0.000015,0.002074,0.000003,0.000363,8.095234e-06,0.000003,0.000003,0.000013,0.000071,0.000496,0.001124,0.000007,0.000003,0.995818
12254,ERR1215687,no_infection,no_infection,0.000334,0.001780,0.000040,0.000024,1.527316e-06,0.000012,0.000004,0.000026,0.000018,0.001426,0.001524,0.000080,0.000003,0.994727
12255,SRR10366893,no_infection,no_infection,0.000074,0.000811,0.000120,0.000029,2.052533e-05,0.000003,0.000033,0.000006,0.000383,0.000088,0.001185,0.000007,0.000004,0.997235


In [ ]:
# Confusion matrix (row-normalized, %, viruses with count > 10)
label_order = counts[counts > 2].index.tolist()
cm_df = cm_df.reindex(index=label_order, columns=label_order)
cm_pct = cm_df.div(cm_df.sum(axis=1), axis=0) * 100
annot = cm_pct.applymap(lambda v: f"{v:.1f}" if v > 0 else "")

plt.figure(figsize=(12, 10))
sns.heatmap(cm_pct, cmap="Blues", cbar=True, annot=annot, fmt="")
plt.title("Confusion Matrix % (count > 10)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/svm_gene_multiclass_hard_rbf_confusion_matrix.pdf", dpi=300)
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import roc_curve, roc_auc_score

binary_scores_path = f"{RESULT_DIR}/{BINARY_RBF_PREFIX}_scores.csv"

# PR (AUPRC) curves for top 5 viruses (exclude no_infection)
plt.figure(figsize=(6, 6))
prevalences = []
for cls in classes_to_plot:
    if cls == "no_infection" or cls not in class_cols:
        continue
    y_true = (scores_df["y_true"] == cls).astype(int).to_numpy()
    y_score = pd.to_numeric(scores_df[cls], errors="coerce").to_numpy()
    mask = ~pd.isna(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        continue
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    prevalences.append(y_true.mean())
    plt.plot(recall, precision, label=f"{cls} (AUPRC={ap:.2f})")

# Binary PR curve (black dashed), shown before baseline
try:
    binary_df = pd.read_csv(binary_scores_path)
    b_y_true = (binary_df["y_true"] == "infected").astype(int)
    b_y_score = binary_df["p_infected"]
    precision_b, recall_b, _ = precision_recall_curve(b_y_true, b_y_score)
    auprc_b = average_precision_score(b_y_true, b_y_score)
    plt.plot(recall_b, precision_b, "k--", label=f"Binary (AUPRC={auprc_b:.2f})")
except Exception:
    pass

if prevalences:
    baseline = max(prevalences)
    plt.hlines(baseline, 0, 1, colors="gray", linestyles="--", label=f"baseline={baseline:.2f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR curves: top 5 viruses")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/svm_gene_multiclass_hard_rbf_pr_top5.pdf", dpi=300)
plt.show()

# ROC curves for the same classes
plt.figure(figsize=(6, 6))
for cls in classes_to_plot:
    if cls == "no_infection" or cls not in class_cols:
        continue
    y_true = (scores_df["y_true"] == cls).astype(int).to_numpy()
    y_score = pd.to_numeric(scores_df[cls], errors="coerce").to_numpy()
    mask = ~pd.isna(y_score)
    y_true = y_true[mask]
    y_score = y_score[mask]
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        continue
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auroc = roc_auc_score(y_true, y_score)
    plt.plot(fpr, tpr, label=f"{cls} (AUROC={auroc:.2f})")

# Binary ROC curve (black dashed), shown before diagonal baseline
try:
    binary_df = pd.read_csv(binary_scores_path)
    b_y_true = (binary_df["y_true"] == "infected").astype(int)
    b_y_score = binary_df["p_infected"]
    fpr_b, tpr_b, _ = roc_curve(b_y_true, b_y_score)
    auroc_b = roc_auc_score(b_y_true, b_y_score)
    plt.plot(fpr_b, tpr_b, "k--", label=f"Binary (AUROC={auroc_b:.2f})")
except Exception:
    pass

plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC curves: top 5 viruses")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/svm_gene_multiclass_hard_rbf_roc_top5.pdf", dpi=300)
plt.show()

In [ ]:
import anndata as ad
ad_sc = ad.read_h5ad(
    DATA_PATH
)
virus_obs = ad_sc.obs.iloc[:, -54:-27].apply(pd.to_numeric, errors="coerce").fillna(0)

In [ ]:
# Confusion matrix with misclassification difference (row-normalized, %, count > 10)
label_order = counts[counts > 2].index.tolist()
cm_df_sub = cm_df.reindex(index=label_order, columns=label_order)
cm_pct = cm_df_sub.div(cm_df_sub.sum(axis=1), axis=0) * 100

# Bracket: ad_sc.obs.iloc[:, -54:-27] counts (same bracket logic as old virus_counts_df)
import scanpy as sc

scores_idx = scores_df.copy()
if "Unnamed: 0" in scores_idx.columns:
    scores_idx = scores_idx.set_index("Unnamed: 0")
elif "index" in scores_idx.columns:
    scores_idx = scores_idx.set_index("index")

annot_plain = cm_pct.copy().astype(str)
annot_bracket_only = cm_pct.copy().astype(str)
cm_bracket = cm_pct.copy().astype(float)

for i in cm_pct.index:
    true_mask = scores_idx["y_true"] == i
    total_true = int(true_mask.sum())
    for j in cm_pct.columns:
        val = cm_pct.loc[i, j]
        if pd.isna(val) or val == 0:
            annot_plain.loc[i, j] = ""
            annot_bracket_only.loc[i, j] = ""
            continue
        if i == j:
            cm_bracket.loc[i, j] = val
            s = f"{val:.1f}"
            annot_plain.loc[i, j] = s
            annot_bracket_only.loc[i, j] = s
            continue
        present_pct = np.nan
        if total_true > 0 and j in virus_obs.columns:
            pred_mask = true_mask & (scores_idx["y_pred"] == j)
            if pred_mask.any():
                run_ids = scores_idx.index[pred_mask]
                col = virus_obs.reindex(run_ids.astype(str))[j].fillna(0)
                present = (col > 0).sum()
                present_pct = (present / total_true) * 100
        s_plain = f"{val:.1f}"
        annot_plain.loc[i, j] = s_plain
        if pd.isna(present_pct) or present_pct == 0:
            cm_bracket.loc[i, j] = val
            annot_bracket_only.loc[i, j] = s_plain
        else:
            cm_bracket.loc[i, j] = val - present_pct
            annot_bracket_only.loc[i, j] = f"{val - present_pct:.1f}"

count_map = scores_df["y_true"].value_counts()
base_labels = [str(lbl) for lbl in label_order]
ytick_labels = [f"{lbl}\n(n={int(count_map.get(lbl, 0))})" for lbl in base_labels]

# Plot 1: row-normalized % only (no bracket)
plt.figure(figsize=(12, 10))
ax1 = sns.heatmap(cm_pct, cmap="Blues", cbar=True, annot=annot_plain, fmt="")
ax1.set_title("Confusion Matrix % (row-normalized)")
ax1.set_xlabel("Predicted")
ax1.set_ylabel("True")
ax1.set_yticklabels(ytick_labels, rotation=0)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/svm_gene_multiclass_hard_rbf_confusion_matrix_rowpct.pdf", dpi=300)
plt.show()

# Plot 2: colors = bracket residual off-diagonal where defined; else row %
plt.figure(figsize=(12, 10))
ax2 = sns.heatmap(
    cm_bracket,
    cmap="Blues",
    cbar=True,
    annot=annot_bracket_only,
    fmt="",
)
ax2.set_title("Confusion Matrix % (row-normalized)")
ax2.set_xlabel("Predicted")
ax2.set_ylabel("True")
ax2.set_yticklabels(ytick_labels, rotation=0)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/svm_gene_multiclass_hard_rbf_confusion_matrix_bracket_term.pdf", dpi=300)
plt.show()

In [ ]:
# Confusion matrix with misclassification difference (row-normalized, %, count > 10)
label_order = counts[counts > 2].index.tolist()
cm_df_sub = cm_df.reindex(index=label_order, columns=label_order)
cm_pct = cm_df_sub.div(cm_df_sub.sum(axis=1), axis=0) * 100

# Bracket: ad_sc.obs.iloc[:, -54:-27] counts (same bracket logic as old virus_counts_df)
import scanpy as sc

scores_idx = scores_df.copy()
if "Unnamed: 0" in scores_idx.columns:
    scores_idx = scores_idx.set_index("Unnamed: 0")
elif "index" in scores_idx.columns:
    scores_idx = scores_idx.set_index("index")

annot = cm_pct.copy().astype(str)
for i in cm_pct.index:
    true_mask = scores_idx["y_true"] == i
    total_true = int(true_mask.sum())
    for j in cm_pct.columns:
        val = cm_pct.loc[i, j]
        if pd.isna(val) or val == 0:
            annot.loc[i, j] = ""
            continue
        if i == j:
            annot.loc[i, j] = f"{val:.1f}"
            continue
        present_pct = np.nan
        if total_true > 0 and j in virus_obs.columns:
            pred_mask = true_mask & (scores_idx["y_pred"] == j)
            if pred_mask.any():
                run_ids = scores_idx.index[pred_mask]
                col = virus_obs.reindex(run_ids.astype(str))[j].fillna(0)
                present = (col > 0).sum()
                present_pct = (present / total_true) * 100
        if pd.isna(present_pct) or present_pct == 0:
            annot.loc[i, j] = f"{val:.1f}"
        else:
            annot.loc[i, j] = f"{val:.1f}\n({val - present_pct:.1f})"

plt.figure(figsize=(12, 10))
ax = sns.heatmap(cm_pct, cmap="Blues", cbar=True, annot=annot, fmt="")
plt.title("Confusion Matrix % (bracket: % − frac true-i with count>0 for virus j)")
plt.xlabel("Predicted")
plt.ylabel("True")

# Add test-set counts to y-axis labels (keep class label)
count_map = scores_df["y_true"].value_counts()
base_labels = [str(lbl) for lbl in label_order]
ytick_labels = [f"{lbl}\n(n={int(count_map.get(lbl, 0))})" for lbl in base_labels]
ax.set_yticklabels(ytick_labels, rotation=0)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/svm_gene_multiclass_hard_rbf_confusion_matrix_diff.pdf", dpi=300)
plt.show()